# 🇵🇪 Notebook 05: Validación con Datos de Perú

**Proyecto:** Sistema de Detección de Phishing con Machine Learning  
**Autor:** [Tu Nombre]  
**Universidad:** [Tu Universidad]  
**Fecha:** 2024-2025

---

## Objetivos

1. ✅ Evaluar modelo en URLs específicas de Perú
2. ✅ Analizar rendimiento en sitios locales
3. ✅ Identificar falsos positivos en sitios legítimos peruanos
4. ✅ Probar detección de phishing dirigido a Perú
5. ✅ Generar recomendaciones específicas

---

## Importancia

Este notebook valida que nuestro modelo funcione correctamente con sitios web peruanos,
tanto legítimos como potenciales intentos de phishing dirigidos a usuarios peruanos.

---

## 1. Importar Librerías

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

# Módulos propios
from feature_extraction import URLFeatureExtractor
from evaluation import ModelEvaluator

# Configuración
pd.set_option('display.max_columns', None)
%matplotlib inline

print("✓ Librerías importadas exitosamente")

## 2. Cargar Mejor Modelo

In [ ]:
# Leer cuál fue el mejor modelo
with open('../results/metrics/best_model.txt', 'r') as f:
    content = f.read()
    best_model_name = content.split('\n')[0].replace('Best Model: ', '')

print(f"Mejor modelo identificado: {best_model_name}")

# Cargar modelo
model_path = f'../models/{best_model_name}.pkl'
best_model = joblib.load(model_path)

# Cargar scaler
scaler = joblib.load('../models/scaler.pkl')

# Cargar features seleccionadas
selected_features = joblib.load('../data/processed/selected_features.pkl')

print(f"✓ Modelo cargado: {model_path}")
print(f"✓ Scaler cargado")
print(f"✓ Features seleccionadas: {len(selected_features)}")

## 3. URLs de Prueba Peruanas

Definimos URLs legítimas de sitios peruanos conocidos para verificar que no sean clasificadas como phishing.

In [ ]:
# URLs legítimas de Perú (sitios oficiales conocidos)
peru_legitimate_urls = {
    'Bancos': [
        'https://www.viabcp.com',
        'https://www.bcp.com.pe',
        'https://interbank.pe',
        'https://www.bbva.pe',
        'https://www.scotiabank.com.pe',
    ],
    'Gobierno': [
        'https://www.gob.pe',
        'https://www.sunat.gob.pe',
        'https://www.reniec.gob.pe',
        'https://www.minsa.gob.pe',
        'https://www.minedu.gob.pe',
    ],
    'Universidades': [
        'https://www.pucp.edu.pe',
        'https://www.uni.edu.pe',
        'https://www.unmsm.edu.pe',
        'https://www.upc.edu.pe',
        'https://www.ulima.edu.pe',
    ],
    'E-commerce': [
        'https://www.falabella.com.pe',
        'https://www.ripley.com.pe',
        'https://www.plazavea.com.pe',
        'https://www.wong.pe',
        'https://www.mercadolibre.com.pe',
    ]
}

# Aplanar el diccionario
all_peru_urls = []
all_categories = []
for category, urls in peru_legitimate_urls.items():
    all_peru_urls.extend(urls)
    all_categories.extend([category] * len(urls))

print(f"Total de URLs legítimas de Perú: {len(all_peru_urls)}")
print(f"\nCategorías:")
for cat, urls in peru_legitimate_urls.items():
    print(f"  - {cat}: {len(urls)} URLs")

## 4. Extraer Features de URLs Peruanas

In [ ]:
# Extraer features
extractor = URLFeatureExtractor()

print("Extrayendo features de URLs peruanas...\n")
df_peru_features = extractor.extract_features_batch(all_peru_urls, show_progress=True)

# Agregar categoría
df_peru_features['category'] = all_categories
df_peru_features['url'] = all_peru_urls

print(f"\n✓ Features extraídas: {df_peru_features.shape}")
print(f"\nPrimeras filas:")
print(df_peru_features[['url', 'category']].head())

## 5. Preprocesar Features

In [ ]:
# Seleccionar solo las features usadas en el modelo
X_peru = df_peru_features[selected_features]

print(f"Features seleccionadas: {X_peru.shape}")

# Normalizar con el scaler entrenado
X_peru_scaled = pd.DataFrame(
    scaler.transform(X_peru),
    columns=X_peru.columns,
    index=X_peru.index
)

print(f"✓ Features normalizadas")

## 6. Predicciones en URLs Peruanas

In [ ]:
# Realizar predicciones
predictions = best_model.predict(X_peru_scaled)

# Obtener probabilidades si está disponible
if hasattr(best_model, 'predict_proba'):
    probabilities = best_model.predict_proba(X_peru_scaled)
    phishing_proba = probabilities[:, 1]  # Probabilidad de ser phishing
else:
    phishing_proba = None

# Agregar predicciones al dataframe
df_peru_features['prediction'] = predictions
df_peru_features['prediction_label'] = df_peru_features['prediction'].map(
    {0: 'Legitimate', 1: 'Phishing'}
)
if phishing_proba is not None:
    df_peru_features['phishing_probability'] = phishing_proba

print("=" * 70)
print("PREDICCIONES EN URLs PERUANAS")
print("=" * 70)
print(f"\nDistribución de predicciones:")
print(df_peru_features['prediction_label'].value_counts())

# Porcentaje
legitimate_pct = (predictions == 0).sum() / len(predictions) * 100
phishing_pct = (predictions == 1).sum() / len(predictions) * 100

print(f"\n  - Legítimas: {(predictions == 0).sum()} ({legitimate_pct:.1f}%)")
print(f"  - Phishing: {(predictions == 1).sum()} ({phishing_pct:.1f}%)")

## 7. Análisis de Falsos Positivos

In [ ]:
# Falsos positivos (URLs legítimas clasificadas como phishing)
false_positives = df_peru_features[df_peru_features['prediction'] == 1]

print("=" * 70)
print("ANÁLISIS DE FALSOS POSITIVOS")
print("=" * 70)

if len(false_positives) > 0:
    print(f"\n⚠ FALSOS POSITIVOS DETECTADOS: {len(false_positives)}")
    print(f"\nURLs legítimas clasificadas incorrectamente como phishing:\n")
    
    for idx, row in false_positives.iterrows():
        prob_str = f" (prob: {row['phishing_probability']:.2%})" if phishing_proba is not None else ""
        print(f"  ❌ {row['url']} [{row['category']}]{prob_str}")
    
    # Análisis por categoría
    print(f"\nFalsos positivos por categoría:")
    print(false_positives['category'].value_counts())
else:
    print(f"\n✅ ¡EXCELENTE! No se detectaron falsos positivos.")
    print(f"   Todas las URLs legítimas fueron clasificadas correctamente.")

## 8. Análisis por Categoría

In [ ]:
# Análisis por categoría
category_analysis = df_peru_features.groupby('category').agg({
    'prediction': ['count', 'sum', 'mean']
})

category_analysis.columns = ['Total', 'Predicted_Phishing', 'Phishing_Rate']
category_analysis['Accuracy'] = 1 - category_analysis['Phishing_Rate']  # Porque todas son legítimas

print("\n" + "=" * 70)
print("ANÁLISIS POR CATEGORÍA")
print("=" * 70)
print("\n", category_analysis)

# Visualización
fig, ax = plt.subplots(figsize=(10, 6))

categories = category_analysis.index
accuracy_by_cat = category_analysis['Accuracy'] * 100

colors = ['#2ecc71' if acc == 100 else '#e74c3c' for acc in accuracy_by_cat]

ax.bar(categories, accuracy_by_cat, color=colors, alpha=0.8)
ax.axhline(y=100, color='green', linestyle='--', label='100% Accuracy', linewidth=2)
ax.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('Accuracy del Modelo por Categoría de Sitios Peruanos', 
             fontsize=14, fontweight='bold')
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3)
ax.legend()

# Agregar valores
for i, v in enumerate(accuracy_by_cat):
    ax.text(i, v + 2, f'{v:.1f}%', ha='center', fontweight='bold')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# Guardar
output_path = '../results/visualizations/peru_validation_by_category.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"\n✓ Visualización guardada: {output_path}")
plt.show()

## 9. URLs Sospechosas Simuladas

Probemos el modelo con URLs que simulan phishing dirigido a usuarios peruanos.

In [ ]:
# URLs sospechosas simuladas (ejemplos de phishing dirigido a Perú)
suspicious_urls = [
    'http://bcp-secure-login.tk/verify',
    'https://interbank-update.ml/account/confirm.php',
    'http://sunat-verificacion.xyz/actualizar-datos',
    'https://www.reniec-pe.com/validar-identidad',
    'http://falabella-premio.ga/ganador.html',
]

print("=" * 70)
print("PRUEBA CON URLs SOSPECHOSAS (Simuladas)")
print("=" * 70)
print("\nEstas URLs simulan intentos de phishing dirigidos a usuarios peruanos:\n")

# Extraer features
df_suspicious = extractor.extract_features_batch(suspicious_urls, show_progress=False)
df_suspicious['url'] = suspicious_urls

# Preprocesar
X_suspicious = df_suspicious[selected_features]
X_suspicious_scaled = pd.DataFrame(
    scaler.transform(X_suspicious),
    columns=X_suspicious.columns
)

# Predecir
suspicious_predictions = best_model.predict(X_suspicious_scaled)
if hasattr(best_model, 'predict_proba'):
    suspicious_proba = best_model.predict_proba(X_suspicious_scaled)[:, 1]
else:
    suspicious_proba = None

# Mostrar resultados
for i, url in enumerate(suspicious_urls):
    pred_label = 'PHISHING' if suspicious_predictions[i] == 1 else 'LEGITIMATE'
    icon = '🚨' if suspicious_predictions[i] == 1 else '✅'
    prob_str = f" (prob: {suspicious_proba[i]:.2%})" if suspicious_proba is not None else ""
    print(f"  {icon} {url}")
    print(f"     → {pred_label}{prob_str}\n")

# Estadísticas
detected = (suspicious_predictions == 1).sum()
detection_rate = detected / len(suspicious_predictions) * 100

print(f"\nTasa de detección: {detected}/{len(suspicious_predictions)} ({detection_rate:.1f}%)")

if detection_rate == 100:
    print("✅ ¡Excelente! Todas las URLs sospechosas fueron detectadas.")
elif detection_rate >= 80:
    print("⚠ Buena detección, pero hay margen de mejora.")
else:
    print("❌ Tasa de detección baja. Requiere optimización.")

## 10. Guardar Resultados de Validación

In [ ]:
# Guardar resultados
results_path = '../results/metrics/peru_validation_results.csv'
df_peru_features.to_csv(results_path, index=False)
print(f"✓ Resultados guardados: {results_path}")

# Guardar resumen
summary_path = '../results/reports/peru_validation_summary.txt'
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write("=" * 70 + "\n")
    f.write("   RESUMEN DE VALIDACIÓN CON URLs PERUANAS\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Modelo utilizado: {best_model_name}\n\n")
    f.write(f"Total de URLs evaluadas: {len(all_peru_urls)}\n")
    f.write(f"URLs clasificadas como legítimas: {(predictions == 0).sum()} ({legitimate_pct:.1f}%)\n")
    f.write(f"Falsos positivos: {len(false_positives)} ({len(false_positives)/len(all_peru_urls)*100:.1f}%)\n\n")
    f.write("Análisis por categoría:\n")
    f.write(str(category_analysis) + "\n\n")
    f.write(f"Tasa de detección en URLs sospechosas: {detection_rate:.1f}%\n")

print(f"✓ Resumen guardado: {summary_path}")

## 11. Recomendaciones Finales

In [ ]:
print("=" * 70)
print("              RECOMENDACIONES FINALES")
print("=" * 70)

print("\n📌 PARA USUARIOS PERUANOS:")
print("  1. Siempre verificar la URL antes de ingresar datos sensibles")
print("  2. Buscar el candado (HTTPS) en la barra del navegador")
print("  3. Desconfiar de URLs con dominios raros (.tk, .ml, .ga, etc.)")
print("  4. Verificar que el dominio sea el oficial del banco/institución")
print("  5. No hacer clic en enlaces de correos o mensajes sospechosos")

print("\n📌 PARA EL MODELO:")
if len(false_positives) > 0:
    print(f"  ⚠ Se detectaron {len(false_positives)} falsos positivos")
    print("  → Considerar ajustar el umbral de clasificación")
    print("  → Reentrenar con más datos de sitios peruanos legítimos")
else:
    print("  ✅ El modelo tiene excelente precisión en sitios peruanos")

if detection_rate < 100:
    print(f"\n  ⚠ Tasa de detección de phishing: {detection_rate:.1f}%")
    print("  → Agregar más ejemplos de phishing dirigido a Perú")
    print("  → Considerar features específicas del contexto peruano")

print("\n📌 PRÓXIMOS PASOS:")
print("  1. Integrar el modelo en una extensión de navegador")
print("  2. Crear API REST para verificación de URLs en tiempo real")
print("  3. Actualizar el modelo periódicamente con nuevos datos")
print("  4. Monitorear rendimiento en producción")
print("  5. Implementar sistema de feedback de usuarios")

print("\n" + "=" * 70)

## 12. Resumen Final del Proyecto

In [ ]:
print("="*70)
print("          RESUMEN FINAL DEL PROYECTO COMPLETO")
print("="*70)

print("\n🎯 OBJETIVO ALCANZADO:")
print("  ✅ Sistema completo de detección de phishing con ML")
print("  ✅ 8 modelos individuales + 2 ensembles entrenados")
print("  ✅ Evaluación exhaustiva con múltiples métricas")
print("  ✅ Validación específica para el contexto peruano")
print("  ✅ Visualizaciones profesionales generadas")
print("  ✅ Documentación completa")

print(f"\n🏆 MEJOR MODELO: {best_model_name}")

print("\n📊 VALIDACIÓN PERÚ:")
print(f"  - URLs evaluadas: {len(all_peru_urls)}")
print(f"  - Accuracy: {legitimate_pct:.1f}%")
print(f"  - Falsos positivos: {len(false_positives)}")
print(f"  - Detección de phishing: {detection_rate:.1f}%")

print("\n📁 ENTREGABLES:")
print("  ✅ Código fuente modular (src/)")
print("  ✅ 6 Jupyter Notebooks completos")
print("  ✅ 10 modelos entrenados guardados")
print("  ✅ Visualizaciones en alta resolución (300 DPI)")
print("  ✅ Reportes y métricas en CSV")
print("  ✅ Documentación (README.md)")

print("\n🎓 LISTO PARA TESIS:")
print("  ✅ Metodología rigurosa aplicada")
print("  ✅ Buenas prácticas de ML implementadas")
print("  ✅ Código bien documentado (PEP 8)")
print("  ✅ Resultados reproducibles (random_state=42)")
print("  ✅ Análisis exhaustivo completado")

print("\n" + "="*70)
print("\n🎉 ¡PROYECTO COMPLETADO EXITOSAMENTE!")
print("\n✨ Listo para presentación de tesis universitaria")
print("="*70)